<a href="https://colab.research.google.com/github/firdoushkhilji/firdoushkhilji-7006SCN_FK_16943920/blob/task6/Task6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
#import and start pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NHS_Prescription_Cost_Prediction") \
    .getOrCreate()

print('Spark Successfully Started!')

Spark Successfully Started!


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import os
os.listdir('/content/drive/MyDrive/')

['TASK_DATASET', 'Colab Notebooks']

In [12]:
os.listdir('/content/drive/MyDrive/TASK_DATASET/')

['epd_snomed_202603.csv',
 'train_data.parquet',
 'test_data.parquet',
 'results_dt.json',
 'results_logreg.json',
 'results_lr.json',
 'results_rf.json',
 'task3_summary_table.csv',
 'roc_pr_curves.png',
 'shap_summary.png',
 'shap_lr_summary.png',
 'shap_dt_summary.png',
 'shap_logreg_summary.png',
 'tableau_model_summary.csv',
 'tableau_data_quality.csv',
 'tableau_feature_importance.csv']

In [13]:
#READ CSV FILE USING SPARK
df = spark.read.csv('/content/drive/MyDrive/TASK_DATASET/epd_snomed_202603.csv', header=True, inferSchema=True)

In [15]:
# Task 6 - Export data for Tableau

# 1. Export model summary metrics for Dashboard 2
import pandas as pd

model_summary = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'Logistic Regression', 'Decision Tree'],
    'Type': ['Regression', 'Regression', 'Classification', 'Classification'],
    'RMSE': [8.23, 115.21, None, None],
    'R2': [0.9988, 0.7400, None, None],
    'MAE': [2.43, 19.40, None, None],
    'Accuracy': [None, None, 0.9720, 0.9929],
    'AUC_ROC': [None, None, 0.9983, 0.9974],
    'F1_Score': [None, None, 0.9712, 0.9929],
    'Training_Time_Seconds': [42.91, 44.76, 49.69, 82.67]
})

# Tableau Public works with CSV files, not Parquet
model_summary.to_csv('/content/drive/MyDrive/TASK_DATASET/tableau_model_summary.csv', index=False)
print("Model summary exported!")
print(model_summary)

Model summary exported!
                 Model            Type    RMSE      R2    MAE  Accuracy  \
0    Linear Regression      Regression    8.23  0.9988   2.43       NaN   
1        Random Forest      Regression  115.21  0.7400  19.40       NaN   
2  Logistic Regression  Classification     NaN     NaN    NaN    0.9720   
3        Decision Tree  Classification     NaN     NaN    NaN    0.9929   

   AUC_ROC  F1_Score  Training_Time_Seconds  
0      NaN       NaN                  42.91  
1      NaN       NaN                  44.76  
2   0.9983    0.9712                  49.69  
3   0.9974    0.9929                  82.67  


In [20]:
#Show null counts, data distributions, partition sizes, ingestion timing

# Data quality metrics from Task 1/2

data_quality = pd.DataFrame({

    'Metric': [

        'Total Rows (Original)', 'Total Columns (Original)',

        'Columns After Cleaning', 'Missing POSTCODE',

        'Missing SNOMED_CODE', 'Rows After Cleaning',

        'File Size GB', 'Partitions Used'

    ],

    'Value': [18364409, 27, 19, 6793, 11449, 18364409, 7.15, 8]

})



data_quality.to_csv('/content/drive/MyDrive/TASK_DATASET/tableau_data_quality.csv', index=False)

print("Data quality metrics exported!")

print(data_quality)

Data quality metrics exported!
                     Metric        Value
0     Total Rows (Original)  18364409.00
1  Total Columns (Original)        27.00
2    Columns After Cleaning        19.00
3          Missing POSTCODE      6793.00
4       Missing SNOMED_CODE     11449.00
5       Rows After Cleaning  18364409.00
6              File Size GB         7.15
7           Partitions Used         8.00


In [21]:
#Show model comparison metrics, ROC/AUC chart, SHAP feature importance rankings

# Random Forest feature importance from Task 3

feature_importance = pd.DataFrame({

    'Feature': ['NIC', 'LOG_NIC', 'ITEMS', 'TOTAL_QUANTITY', 'QUANTITY',

                'SNOMED_CODE', 'LOG_TOTAL_QUANTITY', 'LOG_QUANTITY', 'ADQ_USAGE'],

    'Importance': [0.3914, 0.1925, 0.1294, 0.0433, 0.0410, 0.0248, 0.0201, 0.0080, 0.0037]

})



feature_importance.to_csv('/content/drive/MyDrive/TASK_DATASET/tableau_feature_importance.csv', index=False)

print("Feature importance exported!")

Feature importance exported!


In [23]:
#Turn data findings into actionable recommendations; connect predictions to real-world decisions
# Aggregate actual NHS data by region for business insights dashboard

import pyspark.sql.functions as F

# Aggregate actual NHS data by region for business insights dashboard

regional_summary = train_sample.groupBy('REGIONAL_OFFICE_NAME').agg(

    F.avg('ACTUAL_COST').alias('avg_cost'),

    F.sum('ACTUAL_COST').alias('total_cost'),

    F.count('*').alias('prescription_count'),

    F.sum('HIGH_COST').alias('high_cost_count')

).toPandas()


regional_summary.to_csv('/content/drive/MyDrive/TASK_DATASET/tableau_regional_summary.csv', index=False)

print("Regional summary exported!")

print(regional_summary)

Regional summary exported!
       REGIONAL_OFFICE_NAME   avg_cost    total_cost  prescription_count  \
0           EAST OF ENGLAND  54.141544  263290.32655                4863   
1                    LONDON  42.242443  279391.51785                6614   
2                NORTH WEST  49.411950  319398.84251                6464   
3  NORTH EAST AND YORKSHIRE  50.128468  348643.49623                6955   
4                SOUTH EAST  58.343501  357879.03779                6134   
5                  MIDLANDS  48.757311  423798.54497                8692   
6              UNIDENTIFIED  77.767111     699.90400                   9   
7                SOUTH WEST  52.099992  212932.66870                4087   

   high_cost_count  
0             1003  
1             1107  
2             1156  
3             1350  
4             1223  
5             1680  
6                2  
7              831  
